# Formal Consistency Checking with gamen-validate

cwyde's `cwyde-haskell-bridge` package connects to `gamen-validate`, the Haskell binary from 
[gamen-hs](https://github.com/chapmanbe/gamen-hs). When available, it performs modal-logic 
consistency checking on assertion sets using the full 24-constructor formula language 
(B&D notation). When unavailable, cwyde falls back to a pure-Python YAML table interpreter.

This notebook demonstrates the bridge architecture. The code is runnable in both states.

In [1]:
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

## 1. Discover gamen-validate

`find_gamen_validate()` searches (in order): `CWYDE_GAMEN_BIN` env var, `GAMEN_VALIDATE_BIN` 
env var, system PATH, cabal dist-newstyle build output.

In [2]:
from cwyde_haskell_bridge.discovery import find_gamen_validate

gamen_path = find_gamen_validate()
print(f"gamen-validate: {gamen_path or 'not found (fallback active)'}")

gamen-validate: /Users/brainchapman/Code/Haskell/gamen-hs/dist-newstyle/build/aarch64-osx/ghc-9.8.4/gamen-0.1.0.0/x/gamen-validate/build/gamen-validate/gamen-validate


## 2. Modal formula representation

Each `AssertionCategory` maps to a `ModalFormula` tree. The tree serializes to the JSON format 
that gamen-hs accepts on stdin. `DEFINITE_NEGATED_EXISTENCE` encodes as □¬X (Box(Not(Atom(X)))).

In [3]:
import json
from cwyde.formal.translator import category_to_formula
from cwyde.categories import AssertionCategory

f = category_to_formula(AssertionCategory.DEFINITE_NEGATED_EXISTENCE, "PE")
print(f"Category : DEFINITE_NEGATED_EXISTENCE")
print(f"Formula  : {f}")
print(f"JSON tree: {json.dumps(f.to_tree_json(), indent=2)}")

Category : DEFINITE_NEGATED_EXISTENCE
Formula  : Box(operand=Not(operand=Atom(name='PE')))
JSON tree: {
  "type": "box",
  "operand": {
    "type": "not",
    "operand": {
      "type": "atom",
      "name": "PE"
    }
  }
}


In [4]:
# INDICATION encodes as ¬K_clinician(PE) ∧ ¬K_clinician(¬PE)
f_ind = category_to_formula(AssertionCategory.INDICATION, "PE")
print(f"Category : INDICATION")
print(f"Formula  : {f_ind}")
print(f"JSON tree: {json.dumps(f_ind.to_tree_json(), indent=2)}")

Category : INDICATION
Formula  : Indication(operand=Atom(name='PE'), agent='clinician')
JSON tree: {
  "type": "and",
  "left": {
    "type": "not",
    "operand": {
      "type": "knowledge",
      "agent": "clinician",
      "operand": {
        "type": "atom",
        "name": "PE"
      }
    }
  },
  "right": {
    "type": "not",
    "operand": {
      "type": "knowledge",
      "agent": "clinician",
      "operand": {
        "type": "not",
        "operand": {
          "type": "atom",
          "name": "PE"
        }
      }
    }
  }
}


## 3. Strategy pattern

`CompositeStrategy` wraps `GamenStrategy` (calls the binary) and `FallbackStrategy` (YAML table). 
Infrastructure errors (binary not found, timeout) trigger the fallback. 
Semantic errors (malformed formula) surface immediately — they indicate a translator bug.

In [5]:
from cwyde.formal.strategy import CompositeStrategy, GamenStrategy, FallbackStrategy

strategy = CompositeStrategy(GamenStrategy(), FallbackStrategy())
print(f"CompositeStrategy available : {strategy.is_available()}")
print(f"GamenStrategy available     : {GamenStrategy().is_available()}")
print(f"FallbackStrategy available  : {FallbackStrategy().is_available()}")

CompositeStrategy available : True
GamenStrategy available     : True
FallbackStrategy available  : True


## 4. Live ping and consistency check (gamen only)

If `gamen-validate` is present, we can send formulas directly and receive a consistency verdict. 
The example below checks whether □¬PE (definitely absent) is consistent with ◇PE (possibly present) — 
they are not, since necessity of absence rules out possibility of presence in a normal modal frame.

In [6]:
if gamen_path is not None:
    from cwyde_haskell_bridge import GamenBridge
    bridge = GamenBridge()
    ok = bridge.ping()
    print(f"gamen-validate ping: {'OK' if ok else 'FAILED'}")

    f_neg = category_to_formula(AssertionCategory.DEFINITE_NEGATED_EXISTENCE, "PE")
    f_pos = category_to_formula(AssertionCategory.PROBABLE_EXISTENCE, "PE")
    result = strategy.check_consistency([f_neg, f_pos])
    print(f"Consistency of {{□¬PE, ◇PE}}: {result.consistent}")
    print(f"Explanation: {result.explanation}")
else:
    print("gamen-validate not available — skipping live check.")
    print()
    result = strategy.check_consistency([])
    print(f"Fallback consistency result: {result.consistent}")
    print(f"Fallback explanation: {result.explanation}")
    print()
    print("What gamen would add:")
    print("  - Full modal consistency checking across all entities in a document")
    print("  - Distinguishing ◇¬PE (possibly absent) from □¬PE (necessarily absent)")
    print("  - Detecting contradictions: simultaneous □¬PE and □PE")
    print("  - Reasoning over temporal operators: P(PE) consistent with current □¬PE")

gamen-validate ping: OK
Consistency of {□¬PE, ◇PE}: False
Explanation: 


## 5. What gamen-validate adds

The fallback (`FallbackStrategy`) implements conflict resolution via a YAML precedence table. 
It handles single-entity modifier conflicts (FAMILY vs HISTORICAL, HYPOTHETICAL vs DEFINITE_EXISTENCE) 
but cannot check *cross-entity consistency* within a document.

gamen-validate adds:
- **Modal consistency checking**: detects whether a set of formulas is satisfiable in a Kripke model
- **Cross-entity reasoning**: flags when two findings in the same document contradict each other 
  (e.g., FINDINGS: no PE; IMPRESSION: PE present)
- **Temporal reasoning**: P(PE) (historical) is consistent with current □¬PE; 
  P(¬PE) is consistent with current □PE — the fallback cannot make these distinctions
- **Full 24-constructor support**: stit, xstit, deontic operators for obligation sets 
  (used in `guideline-validation` and `patient-agent-assistant`)

Install gamen-validate with `cabal install gamen-validate` from the 
[gamen-hs repository](https://github.com/chapmanbe/gamen-hs).